# 結果分析とエクスポート

パイプライン実行結果を詳細分析し、Web 表示用の JSON アーティファクトを出力します。

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import sympy
import matplotlib.pyplot as plt

from fusou_formula.pipeline import Pipeline, PipelineConfig
from fusou_formula.data_loader import DataLoader
from fusou_formula.exporter import FormulaExporter
from fusou_formula.validators import FormulaValidator
from fusou_formula.phase5_optimize import RandomRangeSpec, FormulaOptimizer

## 1. パイプライン実行

In [ ]:
loader = DataLoader()
dataset = loader.create_synthetic(
    n_samples=3000,
    formula_str='floor(firepower * 1.5 + 5)',
    noise_range=(0.0, 0.999),
)

config = PipelineConfig(
    sr_niterations=30,
    sr_populations=15,
    optuna_trials=300,
)
pipeline = Pipeline(config)

result = pipeline.run(
    dataset.df, dataset.target_col, dataset.feature_cols,
    category_cols=dataset.category_cols,
)

print(pipeline.report())

## 2. 残差分析

In [ ]:
best = result.best_formula
if best is not None:
    residual_data = FormulaExporter.compute_residuals(
        best, dataset.df, dataset.target_col,
        dataset.feature_cols, result.best_params,
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ヒストグラム
    hist = residual_data['histogram']
    bin_centers = [(hist['bins'][i] + hist['bins'][i+1])/2
                   for i in range(len(hist['counts']))]
    axes[0].bar(bin_centers, hist['counts'],
                width=(hist['bins'][1]-hist['bins'][0])*0.9,
                edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Residual')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Residual Distribution')

    # 残差 vs 入力
    by_input = residual_data['by_input']
    xs = [p['x'] for p in by_input]
    rs = [p['residual'] for p in by_input]
    axes[1].scatter(xs, rs, s=5, alpha=0.5)
    axes[1].axhline(0, color='red', linestyle='--')
    axes[1].set_xlabel(dataset.feature_cols[0])
    axes[1].set_ylabel('Residual')
    axes[1].set_title('Residual vs Primary Feature')

    plt.tight_layout()
    plt.show()
else:
    print('No formula found')

## 3. 予測 vs 実測プロット

In [ ]:
if best is not None:
    y_pred = FormulaOptimizer.evaluate_formula(
        best.expr, dataset.df, dataset.feature_cols, result.best_params,
    )
    y_true = dataset.df[dataset.target_col].values

    valid = np.isfinite(y_pred) & np.isfinite(y_true)

    plt.figure(figsize=(8, 8))
    plt.scatter(y_true[valid], y_pred[valid], s=5, alpha=0.3)
    lim = [min(y_true[valid].min(), y_pred[valid].min()),
           max(y_true[valid].max(), y_pred[valid].max())]
    plt.plot(lim, lim, 'r--', label='Perfect prediction')
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Predicted vs Actual')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. AST ツリー構造（ReactFlow 互換データ）

In [ ]:
from fusou_formula.phase4_ast import ASTMutator

if best is not None:
    ast_tree = ASTMutator.sympy_to_ast_tree(best.expr)
    print(f"Nodes: {len(ast_tree['nodes'])}")
    print(f"Edges: {len(ast_tree['edges'])}")
    print()
    for node in ast_tree['nodes'][:10]:
        print(f"  {node['id']:>10} | {node['data']['label']:<15} type={node['data']['type']}")

## 5. JSON アーティファクトの出力

In [ ]:
exporter = FormulaExporter()

# バリデーション
validator = FormulaValidator(
    random_range=RandomRangeSpec(min_offset=0.0, max_offset=1.0),
)
metrics = validator.evaluate_accuracy(
    best, dataset.df, dataset.target_col,
    dataset.feature_cols, result.best_params,
) if best else None

artifact = exporter.export(
    pipeline_result=result,
    artifact_id='notebook_test_v1',
    target_name='合成テスト (firepower*1.5+5)',
    status='candidate',
    validation_metrics=metrics,
    residual_data=residual_data if best else None,
)

# ローカル保存
filepath = exporter.save(artifact, '../results')
print(f'Saved: {filepath}')

# 中身を確認
print(json.dumps(artifact, indent=2, ensure_ascii=False, default=str)[:2000])

## まとめ

- 残差分析でモデルの適合度を視覚的に確認
- AST ツリーは Web 上で ReactFlow を使って表示
- `results/` に保存された JSON は FUSOU-WEB のローカル開発で直接読み込まれる
- 本番環境にアップロードする場合は `scripts/publish_results.py` を使用